# 4.1 Clean publication data

This notebook does the following:
    - Restricts publication data to consenting researchers

## Set-up

In [ ]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from match_publications_to_labs import match_publications_to_labs

In [ ]:
# Load data
labs = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

sensitive_data = pd.read_csv(
    config.SENSITIVE_DATA / "sensitive_data.csv",
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""], # Only treat empty strings as NaN
    usecols=["labgroupid", "responsible_person"] # Keep only labgroupid and responsible_person columns
)

publications = pd.read_csv(
    config.PUBLICATON_DATA / "1_Raw" / "zora_metadata.csv",
    usecols=["relation", "creator", "title", "date", "subject", "description"]
)

In [3]:
# Merge the datasets on labgroupid
df_consent = pd.merge(labs, sensitive_data, on="labgroupid", how="left")

# Keep only labgroupids where consent to merge
df_consent = df_consent[df_consent["consent_data_merge"] == "Yes I consent to this data collection and merging"]

# Create df with only labgroupid and researcher name
researchers = df_consent[["labgroupid", "responsible_person"]].drop_duplicates()

# Separate researchers into as many cols as needed, based on "," split
researchers["responsible_person"] = researchers["responsible_person"].str.split(",")
max_researchers = researchers["responsible_person"].apply(len).max()
for i in range(max_researchers):
    researchers[f"researcher_{i+1}"] = researchers["responsible_person"].apply(lambda x: x[i] if i < len(x) else None)
researchers = researchers.drop(columns=["responsible_person"])

## (2) Check out publication data

In [11]:
# Print the first 20 rows of creator cols to see format (commented out as sensitive)
# print(publications["creator"].head(20))

## (3) Match publications to labgroupids

In [ ]:
# Reshape researchers (currently wide) into long for matching
researcher_cols = [c for c in researchers.columns if c.startswith("researcher_")]
researchers_long = researchers.melt(
    id_vars="labgroupid", value_vars=researcher_cols, value_name="researcher_name"
).dropna(subset=["researcher_name"])
researchers_long = researchers_long[["labgroupid", "researcher_name"]]

In [ ]:
# Match each publication's creators against the researchers list
matches = match_publications_to_labs(
    publications, 
    id_col="relation", 
    creator_col="creator", 
    researchers_long=researchers_long
)

# Report on matching progress
n_total = len(matches)
n_matched = (matches["matched_labgroupids"].str.len() > 0).sum()
n_ambiguous = (matches["ambiguous_creators"].str.len() > 0).sum()
print(f"Matched {n_matched} of {n_total} publications to at least one labgroupid.")
print(f"{n_ambiguous} publication(s) have an ambiguous creator match (flagged, not dropped).")

In [ ]:
# Merge matches back onto publications
publications = publications.merge(matches, on="relation", how="left")

# Join list columns into delimited strings so they save cleanly to CSV
for col in ["matched_labgroupids", "match_confidence", "ambiguous_creators"]:
    publications[col] = publications[col].apply(
        lambda x: "; ".join(str(v) for v in x) if isinstance(x, list) else x
    )

## Save processed dataset

In [ ]:
# Save processed dataset
publications.to_csv(config.PUBLICATON_DATA / "2_Processed" / "publications_matched.csv", index=False)